<a href="https://colab.research.google.com/github/Western-Windows/SociaLift/blob/engagement/engagement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# 0. Install dependencies (Colab / local with internet)
%pip install -q transformers datasets accelerate sentencepiece scikit-learn torch tqdm

In [7]:
# 1. Imports and configuration

import json
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    get_linear_schedule_with_warmup
)

tqdm.pandas()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [8]:
import json
import numpy as np
import pandas as pd
from tqdm import tqdm

# -------------------------------------------------------------
# 1) Load Brand Data & Build Follower Map
# -------------------------------------------------------------
with open('Facebook_Brand_Data_Analysis.json', 'r', encoding='utf-8') as f:
    brand_metadata = json.load(f)

# Helper function to convert "43M", "~450K" to numeric
def parse_followers(f_str):
    if not isinstance(f_str, str):
        return f_str

    # Remove tilde and commas, make uppercase
    f_str = f_str.replace('~', '').replace(',', '').strip().upper()
    multiplier = 1

    if 'K' in f_str:
        multiplier = 1000
        f_str = f_str.replace('K', '')
    elif 'M' in f_str:
        multiplier = 1000000
        f_str = f_str.replace('M', '')

    try:
        return int(float(f_str) * multiplier)
    except ValueError:
        return np.nan

# Build dictionary matching brand/username to follower count
followers_map = {}
for brand in brand_metadata:
    followers_num = parse_followers(brand.get('Followers', ''))

    username = str(brand.get('Username', '')).lower()
    brand_name = str(brand.get('Brand Name', '')).lower()

    if username: followers_map[username] = followers_num
    if brand_name: followers_map[brand_name] = followers_num

# Aliases to fix slight name mismatches between the two files
aliases = {
    'arrow 1851': 'arrow',
    'clarks shoes': 'clarks',
    'h&m': 'hm',
    'redtape': 'red tape'
}

# -------------------------------------------------------------
# 2) Load Posts Data & Process Rows
# -------------------------------------------------------------
with open('dataset.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

flattened_rows = []

for file_key, doc in tqdm(raw_data.items()):
    posts = doc.get('data', [])
    for post in posts:
        row = {}
        owing_profile = post.get('owing_profile', {})

        # Core Post Details
        row['page_name'] = owing_profile.get('name') or owing_profile.get('short_name')
        row['post_text'] = post.get('context')
        row['date_of_publish'] = post.get('published_date')

        # Pull followers using the map and fallback to our aliases to ensure strict matching
        search_key = str(row['page_name']).lower()
        search_key = aliases.get(search_key, search_key)
        row['page_followers'] = followers_map.get(search_key, np.nan)

        # Sub-reactions mapped from Arabic terms
        sub_reactions = post.get('sub_reactions', {}) or {}
        row['like_count']  = sub_reactions.get('أعجبني', 0)
        row['love_count']  = sub_reactions.get('أحببته', 0)
        row['haha_count']  = sub_reactions.get('هاهاها', 0)
        row['wow_count']   = sub_reactions.get('واااو', 0)
        row['sad_count']   = sub_reactions.get('أحزنني', 0)
        row['angry_count'] = sub_reactions.get('أغضبني', 0)
        row['care_count']  = sub_reactions.get('أدعمه', 0)

        # High-level reaction and share stats
        row['total_reactions'] = post.get('reaction_count.count', 0)
        row['comments_count']  = post.get('comment_rendering_instance.comments.total_count', 0)
        row['share_count']     = post.get('share_count.count', 0)

        flattened_rows.append(row)

posts_df = pd.DataFrame(flattened_rows)

# -------------------------------------------------------------
# 3) Calculate Engagement Scores & Labels
# -------------------------------------------------------------
reaction_weights = {
    'like_count': 1.0,
    'love_count': 2.0,
    'haha_count': 1.5,
    'wow_count': 1.5,
    'sad_count': 0.5,
    'angry_count': 0.5,
    'care_count': 1.5,
    'share_count': 3.0
}

# Ensure numeric columns
numeric_cols = list(reaction_weights.keys()) + ['page_followers']
for col in numeric_cols:
    if col in posts_df.columns:
        posts_df[col] = pd.to_numeric(posts_df[col], errors='coerce').fillna(0)

# Weighted engagement
posts_df['engagement_weighted'] = 0.0
for col, w in reaction_weights.items():
    if col in posts_df.columns:
        posts_df['engagement_weighted'] = posts_df['engagement_weighted'] + w * posts_df[col]

# Engagement rate
followers_series = posts_df['page_followers'].replace(0, pd.NA)
posts_df['engagement_rate'] = posts_df['engagement_weighted'] / followers_series
posts_df['engagement_rate'] = posts_df['engagement_rate'].fillna(0)

# Quantile-based engagement labelling
posts_df['engagement_label'] = pd.qcut(
    posts_df['engagement_rate'],
    q=3,
    labels=['low', 'avg', 'high'],
    duplicates='drop'
)

# Optional log-scaled rate
posts_df['engagement_rate_log'] = np.log1p(posts_df['engagement_rate'].astype(float))

# Save out the completed and verified CSV file
posts_df.to_csv('final_dataset_with_followers.csv', index=False)

print(posts_df[['page_name', 'page_followers', 'engagement_weighted', 'engagement_rate', 'engagement_label']].head())

100%|██████████| 24/24 [00:00<00:00, 19036.18it/s]

  page_name  page_followers  engagement_weighted  engagement_rate  \
0    adidas      43000000.0               1156.0         0.000027   
1    adidas      43000000.0                169.0         0.000004   
2    adidas      43000000.0                252.0         0.000006   
3    adidas      43000000.0                181.0         0.000004   
4    adidas      43000000.0                177.5         0.000004   

  engagement_label  
0              avg  
1              low  
2              low  
3              low  
4              low  



/tmp/ipykernel_449/3387787065.py:124: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  posts_df['engagement_rate'] = posts_df['engagement_rate'].fillna(0)


In [9]:
# 4. Optional translation (only run if you *definitely* need it)

translate = False  # Set to True if you want to translate

if translate:
    translation_model_name = "Helsinki-NLP/opus-mt-mul-en"
    trans_tokenizer = AutoTokenizer.from_pretrained(translation_model_name)
    trans_model = AutoModelForSeq2SeqLM.from_pretrained(translation_model_name).to(device)

    def translate_batch(texts, max_length=128, batch_size=16):
        out = []
        for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
            batch = texts[i:i+batch_size]
            enc = trans_tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length
            ).to(device)
            with torch.no_grad():
                gen = trans_model.generate(**enc, max_length=max_length)
            decoded = trans_tokenizer.batch_decode(gen, skip_special_tokens=True)
            out.extend(decoded)
        return out

    # Using post_text directly if translation is skipped, or creating the translated column
    posts_df["post_text_en"] = translate_batch(posts_df["post_text"].astype(str).tolist())
else:
    # If no translation, we use the original text for the following steps
    posts_df["post_text_en"] = posts_df["post_text"].astype(str)

In [10]:
# 5. Train/validation split and label encoding

label_encoder = LabelEncoder()
posts_df["label_id"] = label_encoder.fit_transform(posts_df["engagement_label"].astype(str))
num_labels = len(label_encoder.classes_)
print("Labels:", list(label_encoder.classes_), "num_labels:", num_labels)

train_df, val_df = train_test_split(
    posts_df,
    test_size=0.2,
    random_state=42,
    stratify=posts_df["label_id"]
)

print("Train size:", len(train_df), "Val size:", len(val_df))

Labels: ['avg', 'high', 'low'] num_labels: 3
Train size: 440 Val size: 111


In [11]:
# 6. TF-IDF + Logistic Regression baseline

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=3
)

X_train_tfidf = tfidf.fit_transform(train_df["post_text_en"])
X_val_tfidf = tfidf.transform(val_df["post_text_en"])

log_reg = LogisticRegression(
    max_iter=200,
    n_jobs=-1,
    class_weight="balanced"
)
log_reg.fit(X_train_tfidf, train_df["label_id"])

val_pred_lr = log_reg.predict(X_val_tfidf)
val_pred_lr_proba = log_reg.predict_proba(X_val_tfidf)

print("TF-IDF + LR accuracy:", accuracy_score(val_df["label_id"], val_pred_lr))
print("TF-IDF + LR macro F1:", f1_score(val_df["label_id"], val_pred_lr, average="macro"))
print(classification_report(val_df["label_id"], val_pred_lr, target_names=label_encoder.classes_))

TF-IDF + LR accuracy: 0.5675675675675675
TF-IDF + LR macro F1: 0.5675816682681671
              precision    recall  f1-score   support

         avg       0.46      0.49      0.47        37
        high       0.62      0.54      0.58        37
         low       0.62      0.68      0.65        37

    accuracy                           0.57       111
   macro avg       0.57      0.57      0.57       111
weighted avg       0.57      0.57      0.57       111



In [12]:
# 7. Torch dataset for transformer models

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "label": int(self.labels[idx])
        }

In [13]:
# 8. Generic training utilities (works for BERT and RoBERTa models)

def collate_fn_builder(tokenizer, max_length=128):
    def collate(batch):
        texts = [item["text"] for item in batch]
        labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
        enc = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        enc["labels"] = labels
        return enc
    return collate

def train_transformer_classifier(
    model,
    tokenizer,
    train_df,
    val_df,
    num_epochs=3,
    batch_size=16,
    max_length=128,
    lr=2e-5,
    warmup_ratio=0.1
):
    train_dataset = TextDataset(train_df["post_text_en"], train_df["label_id"])
    val_dataset = TextDataset(val_df["post_text_en"], val_df["label_id"])

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn_builder(tokenizer, max_length)
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn_builder(tokenizer, max_length)
    )

    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    num_training_steps = num_epochs * len(train_loader)
    num_warmup_steps = int(warmup_ratio * num_training_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )

    best_f1 = 0.0
    best_state = None

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - train"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)

        model.eval()
        all_labels = []
        all_preds = []

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} - val"):
                labels = batch["labels"].numpy()
                all_labels.extend(labels)

                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                logits = outputs.logits
                preds = torch.argmax(logits, dim=-1).cpu().numpy()
                all_preds.extend(preds)

        f1 = f1_score(all_labels, all_preds, average="macro")
        acc = accuracy_score(all_labels, all_preds)
        print(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_acc={acc:.4f}, val_macro_f1={f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    return model, best_f1

In [14]:
# 9. BERT-base classifier with explicit softmax on top

from transformers import AutoModelForSequenceClassification, AutoTokenizer

bert_model_name = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    bert_model_name,
    num_labels=num_labels
)

bert_model, bert_best_f1 = train_transformer_classifier(
    bert_model,
    bert_tokenizer,
    train_df,
    val_df,
    num_epochs=3,
    batch_size=16,
    max_length=128,
    lr=2e-5
)

def bert_predict_with_softmax(texts):
    bert_model.eval()
    bert_model.to(device)
    enc = bert_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs = bert_model(**enc)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    preds = np.argmax(probs, axis=1)
    labels = label_encoder.inverse_transform(preds)
    return labels, probs

val_labels = val_df["label_id"].values
_, bert_probs = bert_predict_with_softmax(val_df["post_text_en"].tolist())
bert_preds = np.argmax(bert_probs, axis=1)

print("BERT accuracy:", accuracy_score(val_labels, bert_preds))
print("BERT macro F1:", f1_score(val_labels, bert_preds, average="macro"))
print(classification_report(val_labels, bert_preds, target_names=label_encoder.classes_))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch 1: train_loss=1.0637, val_acc=0.4595, val_macro_f1=0.4632


Epoch 2/3 - val: 100%|██████████| 7/7 [00:00<00:00,  7.84it/s]


Epoch 2: train_loss=0.9671, val_acc=0.4865, val_macro_f1=0.4811


Epoch 3/3 - val: 100%|██████████| 7/7 [00:00<00:00,  8.44it/s]


Epoch 3: train_loss=0.8653, val_acc=0.5225, val_macro_f1=0.5176
BERT accuracy: 0.5225225225225225
BERT macro F1: 0.5176136126092228
              precision    recall  f1-score   support

         avg       0.48      0.43      0.46        37
        high       0.52      0.68      0.59        37
         low       0.57      0.46      0.51        37

    accuracy                           0.52       111
   macro avg       0.52      0.52      0.52       111
weighted avg       0.52      0.52      0.52       111



In [15]:
# 10. RoBERTa-base classifier with explicit softmax on top

roberta_model_name = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_name)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    roberta_model_name,
    num_labels=num_labels
)

roberta_model, roberta_best_f1 = train_transformer_classifier(
    roberta_model,
    roberta_tokenizer,
    train_df,
    val_df,
    num_epochs=3,
    batch_size=16,
    max_length=128,
    lr=2e-5
)

def roberta_predict_with_softmax(texts):
    roberta_model.eval()
    roberta_model.to(device)
    enc = roberta_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs = roberta_model(**enc)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    preds = np.argmax(probs, axis=1)
    labels = label_encoder.inverse_transform(preds)
    return labels, probs

_, roberta_probs = roberta_predict_with_softmax(val_df["post_text_en"].tolist())
roberta_preds = np.argmax(roberta_probs, axis=1)

print("RoBERTa accuracy:", accuracy_score(val_labels, roberta_preds))
print("RoBERTa macro F1:", f1_score(val_labels, roberta_preds, average="macro"))
print(classification_report(val_labels, roberta_preds, target_names=label_encoder.classes_))

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch 1: train_loss=1.0966, val_acc=0.4775, val_macro_f1=0.3849


Epoch 2/3 - val: 100%|██████████| 7/7 [00:00<00:00,  7.81it/s]


Epoch 2: train_loss=1.0502, val_acc=0.5225, val_macro_f1=0.4808


Epoch 3/3 - val: 100%|██████████| 7/7 [00:00<00:00,  7.79it/s]


Epoch 3: train_loss=0.9192, val_acc=0.5495, val_macro_f1=0.5044
RoBERTa accuracy: 0.5495495495495496
RoBERTa macro F1: 0.5044326241134752
              precision    recall  f1-score   support

         avg       0.55      0.16      0.25        37
        high       0.58      0.68      0.62        37
         low       0.53      0.81      0.64        37

    accuracy                           0.55       111
   macro avg       0.55      0.55      0.50       111
weighted avg       0.55      0.55      0.50       111



In [16]:
# 11. Simple ensemble: average BERT and RoBERTa probabilities

ensemble_probs = (bert_probs + roberta_probs) / 2.0
ensemble_preds = np.argmax(ensemble_probs, axis=1)

print("Ensemble accuracy:", accuracy_score(val_labels, ensemble_preds))
print("Ensemble macro F1:", f1_score(val_labels, ensemble_preds, average="macro"))
print(classification_report(val_labels, ensemble_preds, target_names=label_encoder.classes_))

Ensemble accuracy: 0.5585585585585585
Ensemble macro F1: 0.5473346887206574
              precision    recall  f1-score   support

         avg       0.59      0.35      0.44        37
        high       0.55      0.70      0.62        37
         low       0.55      0.62      0.58        37

    accuracy                           0.56       111
   macro avg       0.56      0.56      0.55       111
weighted avg       0.56      0.56      0.55       111



In [17]:
# 12. Collect metrics

results = []

tfidf_lr_macro_f1 = f1_score(val_df["label_id"], val_pred_lr, average="macro")
tfidf_lr_acc = accuracy_score(val_df["label_id"], val_pred_lr)

bert_macro_f1 = f1_score(val_labels, bert_preds, average="macro")
bert_acc = accuracy_score(val_labels, bert_preds)

roberta_macro_f1 = f1_score(val_labels, roberta_preds, average="macro")
roberta_acc = accuracy_score(val_labels, roberta_preds)

ensemble_macro_f1 = f1_score(val_labels, ensemble_preds, average="macro")
ensemble_acc = accuracy_score(val_labels, ensemble_preds)

results.append(["TF-IDF + LR", tfidf_lr_acc, tfidf_lr_macro_f1])
results.append(["BERT-base", bert_acc, bert_macro_f1])
results.append(["RoBERTa-base", roberta_acc, roberta_macro_f1])
results.append(["BERT + RoBERTa ensemble", ensemble_acc, ensemble_macro_f1])

results_df = pd.DataFrame(
    results,
    columns=["Model", "Val Accuracy", "Val Macro F1"]
)
print(results_df)

                     Model  Val Accuracy  Val Macro F1
0              TF-IDF + LR      0.567568      0.567582
1                BERT-base      0.522523      0.517614
2             RoBERTa-base      0.549550      0.504433
3  BERT + RoBERTa ensemble      0.558559      0.547335
